In [ ]:
import pandas as pd
df = pd.read_csv('data/tag1.csv')
df = df.drop(['Unnamed: 0'], axis = 1)

In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 757 entries, 0 to 756
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   content   757 non-null    object
 1   keywords  757 non-null    object
dtypes: object(2)
memory usage: 12.0+ KB


In [48]:
from fast_langdetect import detect

def detect_my_lang(column):
    unique = []
    for i in column:
        language = detect(i)[0]['lang']
        if language not in unique:
            unique.append(language)
    return unique

In [ ]:
detect_my_lang(df['keywords'])
# In Dutch

['nl']

In [50]:
detect_my_lang(df['content'])
#fr: French nl: Dutch de: German en: This three seems main language......
# English ru: Russian sv: Swedish

['fr', 'nl', 'de', 'en', 'ru', 'sv']

In [ ]:
import re

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text

In [55]:
import spacy

print(spacy.util.get_installed_models())

['en_core_web_md']


In [52]:
temp = df['content'].apply(clean_text)

In [53]:
detect_my_lang(temp) 
# even after the cleaning the 3 language remains occuring frequently.

['fr', 'nl', 'de', 'en', 'sv']

# Model

In [3]:
import pandas as pd
import json

with open('data/content_list.json', 'r') as file:
    content_list = json.load(file)

In [4]:
df = pd.read_csv('data/processed_tag.csv')

In [5]:
df['features'] = pd.Series(content_list)

df = df.dropna()

In [6]:
df['processed_text'] = df['features'].apply(lambda x: " ".join(x))

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [8]:
df['processed_text'] = df['features'].apply(lambda x: " ".join(x))

le = LabelEncoder()
y = le.fit_transform(df['keywords'])

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['processed_text'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [9]:
model = LogisticRegression()
model.fit(X_train, y_train)

# evaluate
preds = model.predict(X_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.93      0.97      0.95       143
           1       0.95      0.87      0.91        84

    accuracy                           0.93       227
   macro avg       0.94      0.92      0.93       227
weighted avg       0.93      0.93      0.93       227



In [10]:
# train
model = LinearSVC()
model.fit(X_train, y_train)

# evaluate
preds = model.predict(X_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       143
           1       0.99      0.98      0.98        84

    accuracy                           0.99       227
   macro avg       0.99      0.98      0.99       227
weighted avg       0.99      0.99      0.99       227



In [11]:
# This much good accuracy looks suspicious so, Checked 
# and Find out there are very high amount of duplicate data.

df.duplicated(subset=['processed_text']).sum()
# Found 351 something duplicated data.

df = df.drop_duplicates(subset=['processed_text'])

In [12]:
le = LabelEncoder()
y = le.fit_transform(df['keywords'])

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['processed_text'])


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y)


In [14]:
model = LogisticRegression(penalty='l2')
model.fit(X_train, y_train)

# evaluate
preds = model.predict(X_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.68      0.99      0.80        77
           1       0.90      0.20      0.33        45

    accuracy                           0.70       122
   macro avg       0.79      0.59      0.57       122
weighted avg       0.76      0.70      0.63       122



In [15]:
# train
model = LinearSVC()
model.fit(X_train, y_train)

# evaluate
preds = model.predict(X_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.85      0.96      0.90        77
           1       0.91      0.71      0.80        45

    accuracy                           0.87       122
   macro avg       0.88      0.84      0.85       122
weighted avg       0.87      0.87      0.86       122



# LSTM

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['processed_text'],
    y,
    test_size=0.3,
    stratify=y
)

In [23]:
from collections import Counter
import numpy as np

all_words = []

for text in X_train:
    all_words.extend(text.split())

counter = Counter(all_words)

freqs = np.array(list(counter.values()))
freqs_sorted = np.sort(freqs)[::-1]

cumulative = np.cumsum(freqs_sorted) / np.sum(freqs_sorted)

vocab_size_95 = np.argmax(cumulative >= 0.96)
# so considering 10000 as vocab size
print("Vocab size for 95% coverage:", vocab_size_95)

Vocab size for 95% coverage: 10576


In [24]:
from keras.preprocessing.text import Tokenizer

vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

train_seq = tokenizer.texts_to_sequences(X_train)
test_seq = tokenizer.texts_to_sequences(X_test)

In [25]:
disti = []
for i in train_seq:
    disti.append(len(i))

import numpy as np

max_len = int(np.percentile(disti, 99))
print("95th percentile max_len:", max_len)

95th percentile max_len: 5773


In [26]:
from keras.preprocessing.sequence import pad_sequences

X_train = pad_sequences(train_seq, maxlen=max_len, padding='post')
X_test = pad_sequences(test_seq, maxlen=max_len, padding='post')

In [27]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_train = torch.tensor(X_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [28]:
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    num_workers=2
)

In [ ]:
str.lowe

In [43]:
class ClassifyGRU(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.rnn = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):

        x = self.embedding(x)

        _, hidden = self.rnn(x)

        hidden = hidden[-1]

        out = self.fc(hidden)

        return out

In [44]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ClassifyGRU(
    vocab_size=vocab_size,
    embed_dim=128,
    hidden_dim=128
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [45]:
model

ClassifyGRU(
  (embedding): Embedding(10000, 128, padding_idx=0)
  (rnn): GRU(128, 128, batch_first=True)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [ ]:
epochs = 5

train_batches = len(train_loader)
test_batches = len(test_loader)

for epoch in range(epochs):

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for x, y in train_loader:

        optimizer.zero_grad(set_to_none=True)

        outputs = model(x)

        loss = criterion(outputs, y)

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * y.size(0)

        preds = outputs.argmax(dim=1)

        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    model.eval()

    test_loss = 0
    correct = 0
    total = 0

    with torch.inference_mode():

        for x, y in test_loader:

            outputs = model(x)

            loss = criterion(outputs, y)

            test_loss += loss.item() * y.size(0)

            preds = outputs.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    test_loss /= total
    test_acc = correct / total


    print(
        f"Epoch {epoch+1} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.4f} | "
        f"Test Loss {test_loss:.4f} | "
        f"Test Acc {test_acc:.4f}"
    )